# L07 · 万物皆正弦 — 用正弦波叠加合成方波（square wave），傅里叶直觉一图彻底建立

**目标**：很多正弦叠加能拼出复杂波形；反过来，任何波形都能拆成正弦成分。

**为什么对 Aurora 重要**：`aurora.audio.transforms` 中的 FFT 和 STFT 做的正是这个拆解——把采样信号还原为各频率（frequency）的幅度和相位（phase，φ），频谱（spectrum）图（spectrogram）的每一列都是一次傅里叶分析（Fourier analysis）的结果。

← **上一课**　[L06 · 欧拉公式 e^{iθ}=cosθ+isinθ](L06_euler.ipynb)

> 上节课学习了 **欧拉公式 e^{iθ}=cosθ+isinθ**：旋转因子是 FFT 的命根子。  
> 本课将探讨 **万物皆正弦**。

## 本课剧情：拆开合唱团的声音

想象一个合唱团在唱歌。你闭上眼睛：能分辨出男高音、女中音、低音炮……每个声部都在唱不同的"音高"，但你的耳朵能把它们一一拆开。

这不是魔法——是**叠加原理**（superposition）：复杂波形 = 多个正弦波相加。

反过来：**任何周期波形都能被正弦波的加权和逼近**。这就是 Fourier 分析的核心直觉。

本课我们从最简单的例子入手：用奇次谐波叠加出**方波（square wave）**。  
项数越多，合成波形的边缘越陡、顶部越平——逐渐"压出"方波的形状。

> 这就是 FFT 的逆过程：FFT 把复杂波形拆成一堆正弦波；这里我们把正弦波重新拼回去，感受这个过程。

## 1. 叠加几个正弦 → 复杂波形

方波为什么用正弦叠加？因为数学告诉我们：

```
方波 ≈ (4/π) · [sin(2πt)/1 + sin(6πt)/3 + sin(10πt)/5 + ...]
```

即：只用**奇次谐波（k = 1, 3, 5, ...）**，每个的权重是 4/(πk)。

三个特点值得先记住，等会儿用手算对答案：
1. **频率**：k 次谐波的频率是基频的 k 倍
2. **权重**：频率越高，贡献越小（1/k 衰减）
3. **叠加结果的值域**：理论上在 (−1, +1) 之间（Gibbs 现象：跳变处峰值约 ±1.09，永远存在）

运行下面的格，观察随谐波项数增加，合成波形逐步逼近方波的动画。

## 实验入口：观察谐波叠加

每增加一个奇次谐波，合成波形的边缘就更陡一点，顶部就更平一点。这一格先画基础的 2 Hz + 5 Hz 叠加，后续格把它推广到任意项数。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.linspace(0, 1, 500)
y = np.sin(2*np.pi*2*t) + 0.5*np.sin(2*np.pi*5*t)
plt.plot(t, y); plt.title('2 Hz + 0.5·(5 Hz)'); plt.show()

## 动手观察：每个分量长什么样

下面先把前几个奇次谐波单独打印出来。注意第 k 个分量的频率是基频的 k 倍，权重是 1/k——频率越高，贡献越小。

In [ ]:
import numpy as np

# 前 5 个奇次谐波在 t=0.25 秒处的瞬时值（基频 1 Hz）
t = 0.25  # 方波首个正半周期的 1/4 位置
print(f"{'阶次':>4}  {'sin(2π×k×t)':>14}  {'权重 1/k':>10}  {'贡献':>10}")
print('-' * 48)
for k in range(1, 10, 2):  # 奇次：1,3,5,7,9
    raw = np.sin(2 * np.pi * k * t)
    weighted = raw / k
    print(f'{k:>4}次  {raw:>+14.4f}  {1/k:>10.4f}  {weighted:>+10.4f}')


## 代码实验：叠加项数从 1 到 9

下面从 n=1 到 n=9 逐步增加谐波，观察合成波形的方波近似程度如何随项数提升。

In [ ]:
import numpy as np

# 逐步叠加奇次谐波，观察合成值向 π/4 收敛（t=0.25）
t = 0.25
theory = np.pi / 4  # 方波在 t=0.25 的理论值
acc = 0.0
print(f"{'项数':>4}  {'新加谐波':>6}  {'新增贡献':>12}  {'合成值':>12}  {'误差':>10}")
print('-' * 55)
for i, k in enumerate([1, 3, 5, 7, 9]):
    term = np.sin(2 * np.pi * k * t) / k
    acc += term
    print(f'{i+1:>4}  k={k:1d}次  {term:>+12.4f}  {acc:>+12.4f}  {abs(acc-theory):>10.4f}')
print(f'\nπ/4 ≈ {theory:.4f} （越来越近！）')


## 2. ✏️ 用奇次谐波叠加逼近方波 `square_approx(t, n)`

方波 ≈ `Σ_{k=1,3,5,...} sin(2π·k·t) / k`（取前 n 个奇次谐波）。

**推理路线**：
1. 输出需与 `t` 等长，先用 `np.zeros_like(t)` 初始化累加器 `result`，确保 dtype 和 shape 自动对齐。
2. 奇次谐波下标是 `1, 3, 5, ..., 2n-1`，用 `range(1, 2*n, 2)` 生成，共 `n` 项。
3. 循环内每次把 `np.sin(2*np.pi*k*t)/k` 加到 `result`——分母 `k` 正是让高频分量权重递减的关键，缺掉它级数不收敛。

**参考输入输出**：`n=1` 时只有基频，输出是标准正弦；`n=5` 时波形顶部已出现明显的"肩膀"；`n=50` 时方波的上下平台清晰可见，只在跳变处有吉布斯振铃。

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先明确三件事

写 `square_approx` 前明确三件事：
- 输入：`t`（时间数组）、`n`（叠加的谐波个数）
- 关键步骤：对 `k in [1, 3, 5, ..., 2n-1]` 累加 `sin(2π·k·t) / k`
- 返回：与 `t` 等长的一维数组，值域约为 `[-1, 1]`（Gibbs 过冲约 9%，峰值约 ±1.09）

In [ ]:
import numpy as np
def square_approx(t, n):
    # 傅里叶级数：(4/πk)·sin(kx)；奇次谐波 k=1,3,5...；截断后峰值 ≈ 1（Gibbs 过冲 ~9%）
    # ✏️ TODO: 累加前 n 个奇次谐波 (4/πk)·sin(2πkt)，k=1,3,5,...,2n-1
    raise NotImplementedError("TODO: 累加奇次谐波")


In [ ]:
import numpy as np, matplotlib.pyplot as plt
t = np.linspace(0, 1, 1000)
for n in [1, 3, 10]:
    plt.plot(t, square_approx(t, n), label=f'{n} harmonics')
plt.legend(); plt.title('harmonics build a square wave'); plt.show()
# 自检: 谐波越多, 越接近 ±π/4 的方波幅度
approx = square_approx(t, 50)
# 方波峰值应在 [0.85, 1.15]；若 < 0.85 系数太小，若 > 1.2 可能用了错误比例
assert 0.85 < approx.max() < 1.15, f'方波峰值应接近 1（含 Gibbs 过冲），实际 {approx.max():.3f}'
print('✅ 通过：你亲手用正弦拼出了方波 = 傅里叶级数。')

**🔗 Aurora 连接**：傅里叶变换把信号拆回这些正弦成分(频谱)；L37-L39 的 FFT、L43-L45 的频谱图，做的就是这件事。

**🎉 复数与三角地基齐了**：L04–L07 建立了正弦波、复数、欧拉公式、傅里叶级数四块基础，FFT 所需的数学已就位。线代（L09–L21）、微积分（L22–L26）、概率（L27–L31）将在后续模块依次展开。

In [ ]:
# 小检查：打印方波前 5 个奇次谐波的频率和幅度
# （这就是 L37-L39 FFT 会「看见」的频谱峰）
import numpy as np
f0 = 440  # 基频 A4
print(f"{'阶次':>4}  {'频率':>9}  {'幅度':>8}  {'说明'}")
print("-" * 42)
for k in range(1, 10, 2):  # 奇次：1,3,5,7,9
    freq = k * f0
    amp  = 4 / (np.pi * k)
    print(f"{k:>4}次  {freq:>8.0f} Hz  {amp:>8.4f}  = 4/(π×{k})")
print()
print("规律：只有奇次谐波；幅度按 1/k 递减。")
print("→ 在 L37 FFT 谱里，这些频率位置会出现对应高度的峰值。")


## 参数实验：只改一个旋钮

把 `n` 从 1 增加到 20，观察方波顶部的平坦度如何提高。注意 `t=0.5` 附近的跳变处：即使 `n` 很大，该处的超调量始终接近 9%吉布斯现象（Gibbs phenomenon），增加 `n` 无法消除这个振铃，只能使其向跳变点收缩。

In [ ]:
import numpy as np

# 参数实验：谐波项数 n 越大，t=0.25 处近似越精确
t = 0.25
theory = np.pi / 4
print(f"{'n':>5}  {'近似值':>12}  {'误差':>10}")
for n in [1, 3, 5, 10, 20, 50]:
    val = sum(np.sin(2*np.pi*k*t)/k for k in range(1, 2*n, 2))
    print(f'{n:>5}  {val:>+12.6f}  {abs(val-theory):>10.6f}')
print(f'\nπ/4 ≈ {theory:.6f}')
print('吉布斯现象：即使 n 很大，跳变点附近误差不会完全消失（约 9%）。')


## 本课收束

现在能用 `square_approx(t, n)` 生成任意项数的谐波叠加，并观察合成波形随 n 收敛的过程。这直接对应 `aurora.audio.transforms` 的工作原理：FFT 输出的每个频率桶是某个正弦分量的幅度和相位。STFT 在每个时间帧重复一次这个拆解，得到频谱图的一列。L37-L39 实现离散 FFT 时，会用方波信号验证频谱中只有奇次谐波出现。

下一课（L08）将通过 matplotlib 动画演示复数平面上的单位圆旋转、共轭对称与相位概念，把傅里叶直觉映射到可视化图像。

## ✏️ 白板挑战：傅里叶级数手算（目标 10 分钟）

盖上屏幕，纸上作答。公式：`square_approx(t, n) = (4/π) · Σ_{k=1,3,5,...,2n-1} sin(2π·k·t) / k`

**问 1**：n=1 时（只有 k=1 项），`square_approx(0.25, 1)` 等于多少？
提示：`(4/π) · sin(2π·1·0.25) = (4/π) · sin(π/2) = ?`

**问 2**：理论上，当 n→∞ 时，`square_approx(0.25, n)` 收敛到多少？
（方波在 t=0.25 处的真实值是 +1）

**问 3**：n=2 时（k=1 和 k=3），`square_approx(0.25, 2)` 等于多少？
提示：加上 k=3 的贡献：`(4/π) · sin(2π·3·0.25) / 3 = (4/3π) · sin(3π/2) = ?`（注意方向！）

**问 4**：在 t=0.5（方波的跳变点），任意有限项 n 的叠加结果是多少？为什么？
提示：所有正弦 `sin(2π·k·0.5) = sin(kπ) = 0`（k 为整数）。

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：n=1 时 t=0.25
n1 = (4/np.pi) * np.sin(2*np.pi * 1 * 0.25) / 1
assert np.isclose(n1, 4/np.pi, atol=1e-10)
print(f"Q1 ✅  n=1, t=0.25: (4/π)·sin(π/2) = 4/π ≈ {n1:.4f}")

# 验证用 square_approx
try:
    t_arr = np.array([0.25])
    q1_code = square_approx(t_arr, 1)[0]
    assert np.isclose(q1_code, n1, atol=1e-10), f"square_approx 值 {q1_code:.4f} 与手算 {n1:.4f} 不符"
    print(f"       square_approx(0.25, 1) = {q1_code:.4f}  ✅ 与手算一致")
except NotImplementedError:
    print("       (square_approx 尚未实现，先手算对答案)")

# 问2：理论极限
print(f"\nQ2 ✅  n→∞ 时收敛到 1.0（方波真实值），Gibbs 现象导致跳变处超调约 ±1.09")

# 问3：n=2 时 t=0.25
k1_contrib = (4/np.pi) * np.sin(2*np.pi * 1 * 0.25) / 1
k3_contrib = (4/np.pi) * np.sin(2*np.pi * 3 * 0.25) / 3  # = (4/3π)·sin(3π/2) = -4/(3π)
n2 = k1_contrib + k3_contrib
print(f"\nQ3 ✅  n=2, t=0.25: {k1_contrib:.4f} + ({k3_contrib:.4f}) = {n2:.4f}")
print(f"       k=1 贡献正 4/π，k=3 贡献负 4/(3π)，部分相消后更接近 1")

# 问4：t=0.5 跳变点
for n in [1, 3, 5, 10]:
    try:
        val_at_half = square_approx(np.array([0.5]), n)[0]
        assert np.isclose(val_at_half, 0.0, atol=1e-10), f"n={n}: 期望0，得到{val_at_half}"
    except NotImplementedError:
        # 手算验证
        val_at_half = sum((4/(np.pi*k)) * np.sin(2*np.pi*k*0.5) for k in range(1, 2*n, 2))
        assert abs(val_at_half) < 1e-10
print(f"\nQ4 ✅  任意 n，t=0.5 处叠加结果 = 0（sin(kπ)=0 对所有整数k成立）")
print("\n🎉 傅里叶直觉白板挑战通过！正弦叠加=方波的手算已内化。")

In [ ]:
# ✏️ 本课自评
l07_review = {
    "square_approx_implemented": None,  # square_approx 实现并通过断言？True/False
    "harmonic_series_understood": None, # 理解奇次谐波 + 1/k 权重的傅里叶级数结构？True/False
    "gibbs_phenomenon_intuition": None, # 知道 Gibbs 现象（跳变处超调≈9%）？True/False
    "whiteboard_passed":          None, # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l07_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l07_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L07 全部通关！进入 L08：复数平面可视化')

---

→ **下一课**　[L08 · 复数平面可视化](L08_visual_complex.ipynb)

> 下节课将学习 **复数平面可视化**：单位圆旋转、共轭与相位，matplotlib 动态演示。